# 12 x 75 mm tube height test in Hamilton 32-position carrier

This notebook uses `hamilton_tube_carrier_32_a00` at rail 19 to verify the measured rim height of a 12 x 75 mm test tube in the frontmost carrier site (site 0). The top of the tube is modeled as **25.6 mm above the tube-carrier top plane**.

A 1000 uL filtered tip from A1 of the rack in tip-carrier position 0 at rail 25 moves to the tube center with adjustable left/right and up/down offsets. It stops 5 mm above the expected rim, lowers to the rim, then moves through a halfway checkpoint to the calculated tube bottom.

**Place exactly one 12 x 75 mm tube in site 0, keep every other carrier site empty, keep the deck clear, supervise every move, and execute the cells from top to bottom once.** Do not rerun the deck-layout cell after setup because it creates a new, uninitialized `LiquidHandler`.

In [ ]:
%load_ext autoreload
%autoreload 2

from pylabrobot.liquid_handling import LiquidHandler
from pylabrobot.liquid_handling.backends import STARBackend
from pylabrobot.resources import hamilton_96_tiprack_1000uL_filter
from pylabrobot.resources.hamilton import (
    STARLetDeck,
    TIP_CAR_480_A00,
    hamilton_tube_carrier_32_a00,
)


In [ ]:
# PyLabRobot channels, carrier positions, and carrier sites are zero-based.
CHANNEL = 0  # physical channel 1
TUBE_SITE = 0  # frontmost site
TIP_CARRIER_POSITION = 0
TIP_POSITION = "A1"
TUBE_CARRIER_RAIL = 19
TIP_CARRIER_RAIL = 25

TUBE_DIAMETER_MM = 12.0
TUBE_LENGTH_MM = 75.0
TUBE_TOP_ABOVE_CARRIER_MM = 25.6
ABOVE_RIM_CLEARANCE_MM = 5.0
SAFE_TRAVERSE_Z = 247.6

# Adjust these to center the tip in the physical tube.
# Positive LEFT_RIGHT_OFFSET_MM moves toward increasing deck X; negative moves toward decreasing X.
# Positive UP_DOWN_OFFSET_MM moves toward the rear/increasing deck Y; negative moves toward the front/decreasing Y.
LEFT_RIGHT_OFFSET_MM = 0.0
UP_DOWN_OFFSET_MM = 0.0

# The verified Z heights can still be shifted together if future calibration requires it.
Z_CALIBRATION_OFFSET_MM = 0.0

backend = STARBackend()
lh = LiquidHandler(backend=backend, deck=STARLetDeck())

tip_carrier = TIP_CAR_480_A00(name="tip_carrier")
tip_rack_1000ul = hamilton_96_tiprack_1000uL_filter(
    name="tips_1000ul_filter"
)
tip_carrier[TIP_CARRIER_POSITION].assign_child_resource(tip_rack_1000ul)
lh.deck.assign_child_resource(tip_carrier, rails=TIP_CARRIER_RAIL)

tube_carrier = hamilton_tube_carrier_32_a00(name="tube_carrier")
lh.deck.assign_child_resource(tube_carrier, rails=TUBE_CARRIER_RAIL)


In [ ]:
# Derive and validate all motion targets before connecting to the robot.
site_center = tube_carrier[TUBE_SITE].get_absolute_location(x="c", y="c")
carrier_top_z = tube_carrier.get_absolute_location(z="t").z
expected_rim_z = carrier_top_z + TUBE_TOP_ABOVE_CARRIER_MM
above_rim_z = expected_rim_z + ABOVE_RIM_CLEARANCE_MM
tube_bottom_z = expected_rim_z - TUBE_LENGTH_MM
half_depth_z = tube_bottom_z + TUBE_LENGTH_MM / 2

assert TUBE_SITE == min(tube_carrier.sites), "Site 0 must be frontmost"
assert TUBE_DIAMETER_MM <= tube_carrier[TUBE_SITE].get_size_x()
assert TUBE_DIAMETER_MM <= tube_carrier[TUBE_SITE].get_size_y()
assert tube_carrier.get_size_z() == 60.0
assert abs(carrier_top_z - 160.0) < 0.01
assert abs(expected_rim_z - 185.6) < 0.01
assert tube_bottom_z < half_depth_z < carrier_top_z < expected_rim_z < above_rim_z
assert above_rim_z < SAFE_TRAVERSE_Z
assert tip_rack_1000ul[TIP_POSITION][0].has_tip()
selected_tip = tip_rack_1000ul[TIP_POSITION][0].get_tip()
assert selected_tip.nominal_volume == 1000
assert selected_tip.has_filter

print(lh.deck.summary())
print("\n=== 12 x 75 mm TUBE HEIGHT TARGETS (absolute deck coordinates, mm) ===")
print(f"Tube site center:           X={site_center.x:.2f}, Y={site_center.y:.2f}")
print(f"Carrier top:               Z={carrier_top_z:.2f}")
print(f"Tube above carrier:          {TUBE_TOP_ABOVE_CARRIER_MM:.2f}")
print(f"EXPECTED TUBE RIM:         Z={expected_rim_z:.2f}")
print(f"Above-rim approach:        Z={above_rim_z:.2f}")
print(f"Safe traverse:             Z={SAFE_TRAVERSE_Z:.2f}")
print(f"Half-depth checkpoint:     Z={half_depth_z:.2f}")
print(f"CALCULATED TUBE BOTTOM:    Z={tube_bottom_z:.2f}")
print(f"Tube dimensions:             {TUBE_DIAMETER_MM:.1f} x {TUBE_LENGTH_MM:.1f}")
print(f"Carrier-site opening:        {tube_carrier[TUBE_SITE].get_size_x():.1f} mm")


## Connect and run

Confirm the printed deck layout matches the physical deck. Rail 19 must contain the bare 32-position carrier with one 12 x 75 mm tube in frontmost site 0. Rail 25 must contain the tip carrier with a 1000 uL filtered-tip rack in position 0 and a tip at A1.

In [ ]:
if lh.setup_finished:
    print("Liquid handler setup is already complete; setup was not repeated.")
else:
    await lh.setup(skip_autoload=True)


In [ ]:
def confirm(message: str):
    response = input(f"\n{message} Type GO and press Enter to continue: ").strip().upper()
    if response != "GO":
        raise RuntimeError("Operator cancelled the tube-height test.")


async def run_12x75mm_tube_height_test(
    left_right_offset_mm: float = LEFT_RIGHT_OFFSET_MM,
    up_down_offset_mm: float = UP_DOWN_OFFSET_MM,
    z_offset_mm: float = Z_CALIBRATION_OFFSET_MM,
):
    target_x = site_center.x + left_right_offset_mm
    target_y = site_center.y + up_down_offset_mm
    rim_z = expected_rim_z + z_offset_mm
    approach_z = rim_z + ABOVE_RIM_CLEARANCE_MM
    bottom_z = tube_bottom_z + z_offset_mm
    halfway_z = bottom_z + TUBE_LENGTH_MM / 2
    assert bottom_z < halfway_z < rim_z < approach_z < SAFE_TRAVERSE_Z
    tip_is_attached = False

    print("\n=== CALIBRATED TARGETS (mm) ===")
    print(
        f"Offsets: left/right={left_right_offset_mm:+.2f}, "
        f"up/down={up_down_offset_mm:+.2f}, Z={z_offset_mm:+.2f}"
    )
    print(f"Tube center: X={target_x:.2f}, Y={target_y:.2f}")
    print(f"Tube rim: Z={rim_z:.2f}; approach: Z={approach_z:.2f}")
    print(f"Half-depth: Z={halfway_z:.2f}; tube bottom: Z={bottom_z:.2f}")

    confirm(
        "Verify the 12 x 75 mm tube is in frontmost carrier site 0 at rail 19; "
        "all other tube sites are empty; the 1000 uL filtered tip is at "
        "tip-carrier position 0, A1; deck clear."
    )
    await lh.pick_up_tips(tip_rack_1000ul[TIP_POSITION], use_channels=[CHANNEL])
    tip_is_attached = True

    try:
        await lh.prepare_for_manual_channel_operation(CHANNEL)
        await lh.backend.move_channel_tool_z(channel_idx=CHANNEL, z=SAFE_TRAVERSE_Z)

        confirm(
            f"Ready to traverse to tube center X={target_x:.2f}, Y={target_y:.2f}."
        )
        await lh.move_channel_x(CHANNEL, target_x)
        await lh.move_channel_y(CHANNEL, target_y)

        confirm(
            f"Ready to lower to Z={approach_z:.2f}, "
            f"{ABOVE_RIM_CLEARANCE_MM:.1f} mm above the expected tube rim."
        )
        await lh.backend.move_channel_tool_z(channel_idx=CHANNEL, z=approach_z)
        print("Inspect X/Y centering over the tube before continuing.")

        confirm(
            f"Ready to lower only to the expected tube-rim plane at Z={rim_z:.2f}."
        )
        await lh.backend.move_channel_tool_z(channel_idx=CHANNEL, z=rim_z)
        print("Tip end is at the expected tube-rim height.")

        confirm(
            f"Verify the tip is centered inside the tube. Ready to descend to "
            f"the halfway checkpoint at Z={halfway_z:.2f}."
        )
        await lh.backend.move_channel_tool_z(channel_idx=CHANNEL, z=halfway_z)

        confirm(
            f"Ready for the final descent to the calculated tube bottom at "
            f"Z={bottom_z:.2f}. This target is rim Z minus the 75.0 mm tube length."
        )
        await lh.backend.move_channel_tool_z(channel_idx=CHANNEL, z=bottom_z)
        print("Calculated tube-bottom target reached.")

        confirm("Bottom inspection complete; ready to raise and return the tip.")
        await lh.backend.move_channel_tool_z(channel_idx=CHANNEL, z=SAFE_TRAVERSE_Z)
        await lh.drop_tips(tip_rack_1000ul[TIP_POSITION], use_channels=[CHANNEL])
        tip_is_attached = False
        print(f"Tube-height test complete; tip returned to {TIP_POSITION}.")
    finally:
        if tip_is_attached:
            print("Test interrupted with a tip attached; attempting a safe raise.")
            try:
                await lh.backend.move_channel_tool_z(
                    channel_idx=CHANNEL, z=SAFE_TRAVERSE_Z
                )
                print("Channel raised. Run the recovery cell to return the tip.")
            except Exception as recovery_error:
                print(f"Automatic raise failed: {recovery_error!r}")
                print("Recover the channel manually before moving deck hardware.")


In [ ]:
await run_12x75mm_tube_height_test(
    left_right_offset_mm=LEFT_RIGHT_OFFSET_MM,
    up_down_offset_mm=UP_DOWN_OFFSET_MM,
    z_offset_mm=Z_CALIBRATION_OFFSET_MM,
)


## Recovery

Run this only if the test was interrupted with the 1000 uL tip still attached.

In [ ]:
await lh.backend.move_channel_tool_z(channel_idx=CHANNEL, z=SAFE_TRAVERSE_Z)
await lh.drop_tips(tip_rack_1000ul[TIP_POSITION], use_channels=[CHANNEL])


## Stop

Run after the test or recovery is complete.

In [ ]:
await lh.stop()
